In [ ]:
#This code will wrangle the data obtained from both the TCGA directly and also from related studies applied to this project
#we will need to organize all the tcga IDs from the X data set or all_cases TCGA csv file
#Obtain the gene expression profiles from each of these IDs 
#then create a matrix for each labeled X and Y

#this section is for installing all the necessary modules within the bluebear jupyter environment
#you will need to restart the kernal after install
#!pip install --user pandas
#!pip install --user openpyxl
#!pip install --user pybiomart
#!pip install --user --upgrade attrs
#!pip install --user --upgrade attrs cattrs

In [1]:


#Unique Cases in all_cases tscv file
#This chunk of code will create a df for the unique cases in the all_cases csv file
#to cross reference these IDs to their corresponding gene expression profiles (Y)

#lets import all the necessary modules/add ons
import pandas as pd

#-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#first lets make a list of all unique rows in the case column within the all_cases file

#define the path of the all_cases file
path_all_cases = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/all_cases_ascat2_TCGA.csv'

#load the data into a pandas DataFrame
df_cases = pd.read_csv(path_all_cases)

#check if column names match exact column containing the TCGA IDs
print("Columns in all_cases:", df_cases.columns.tolist())
#print(df_cases.head(2))

#lets filter only for LUSC Lung Squamous Cell Carcinoma and the IDs contain -01A or primary tumor sample
df_cases_LUSC = df_cases[df_cases['Tumor_Type'].str.contains('TCGA-LUSC', na=False)]
df_cases_LUSC = df_cases_LUSC[df_cases_LUSC['Sample_ID'].str.contains('-01A', na=False)]


print(df_cases_LUSC.head())


#then extract unique case IDs from the X/copy-number dataset
unique_cases_x = df_cases_LUSC['Case'].unique()
print(f"Total unique cases found in X data: {len(unique_cases_x)}")
print(unique_cases_x)

#now we will create a standardization function to match TCGA barcode formats
def get_tcga_patient_id(barcode):
    """Truncates a TCGA barcode to the standard 12-character patient ID, 
        handling trailing whitespaces or messy strings."""
    if isinstance(barcode, str):
        return barcode.strip()[:12]
    return barcode

#convert unique_cases_x into a set of standardized patient IDs for lookups
target_patients = set(get_tcga_patient_id(case) for case in unique_cases_x)



Columns in all_cases: ['GDC_Aliquot', 'Chromosome', 'Start', 'End', 'Copy_Number', 'Major_Copy_Number', 'Minor_Copy_Number', 'Case', 'Tumor_Type', 'Sample_ID', 'Sample_UUID', 'Tissue_Type']
                               GDC_Aliquot Chromosome     Start         End  \
8041  12031677-4ed7-42ac-a61a-3866b5a1cba1       chr1     62920   7929506.0   
8042  12031677-4ed7-42ac-a61a-3866b5a1cba1       chr1   7929870  15562377.0   
8043  12031677-4ed7-42ac-a61a-3866b5a1cba1       chr1  15568112  15592573.0   
8044  12031677-4ed7-42ac-a61a-3866b5a1cba1       chr1  15595360  17836417.0   
8045  12031677-4ed7-42ac-a61a-3866b5a1cba1       chr1  17837139  18364119.0   

      Copy_Number  Major_Copy_Number  Minor_Copy_Number          Case  \
8041            2                  2                  0  TCGA-77-8009   
8042            3                  3                  0  TCGA-77-8009   
8043            5                  5                  0  TCGA-77-8009   
8044            3                  3       

In [2]:
#This next chunk of code will use the tcga IDs from the copy number target patients df to match with the Y RNAseq gene expression file
#taken from xenabrowser https://xenabrowser.net/datapages/?hub=https://tcga.xenahubs.net:443 TCGA LUSC RNAseq gene expression

import pandas as pd
#define the path of the rna_seq file
rna_seq_path = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/Y/HiSeqV2'

#load in and visualize gene expression df
df_exp = pd.read_csv(rna_seq_path, sep='\t')
#print(df_exp.head())

#now we need to take the sample names from the column names of the rna seq file
#find the unique names then match them to the target_patients list

#check if column names match exact column containing the TCGA IDs
#print("Columns in all_cases:", df_exp.columns.tolist())
#print(df_exp.head(2))

#we will need to have all the sample IDs have -01 on the end for a primary tissue sample 

#separate sample ID column name from patient ID
sample_col = 'sample' if 'sample' in df_exp.columns else None

#then filter columns and keep sample name column and any column ending in -01 for primary tumor samples
tumor_columns = [col for col in df_exp.columns if col.endswith('-01') or col == sample_col]

df_exp_tumor_only = df_exp[tumor_columns]

#print(df_exp_tumor_only)
#next we will take the column names of the expression dataframe
#first though lets remove the last three characters (-01) of each column name (starting from column 1 not the sample column)

#now in the tumor columns lets remove the last 3 characters (-01)
#then intersect the IDs with the target_patients df

#create a mapping dictionary to rename the columns from full sample IDs to 12-str patient IDs
rename_mapping = {}
for col in df_exp_tumor_only.columns:
    if col != 'sample':
        #strip the last 3 characters (-01) to get the 12-char patient ID
        patient_id = col[:-3]
        rename_mapping[col] = patient_id

#apply the rename to the expression DataFrame
df_exp_pt = df_exp_tumor_only.rename(columns=rename_mapping)

#extract the unique patient IDs present in the expression matrix
exp_patients_set = set(rename_mapping.values())

#since target_patients is a set, we can find the intersection patients present in X and Y
common_patients = target_patients.intersection(exp_patients_set)

#print(common_patients)

#sorted common patients
sorted_common_patients = sorted(list(common_patients))

#subset X and Y data frames

#keep sample and patient ID columns for gene_expression Y df
cols_to_keep = ['sample'] + sorted_common_patients if 'sample' in df_exp_pt.columns else sorted_common_patients
df_exp_final = df_exp_pt[[c for c in cols_to_keep if c in df_exp_pt.columns]]

#create a standard 12-char patient ID column in LUSC copy number df
df_cases_LUSC['clean_case'] = df_cases_LUSC['Case'].apply(get_tcga_patient_id)

#filter out df_cases_LUSc to keep only rows corresponding to sorted_common_patients
df_X_final = df_cases_LUSC[df_cases_LUSC['clean_case'].isin(sorted_common_patients)].copy()

#lets sort for downstream estimators by clean_case and chromosome start
df_X_final = df_X_final.sort_values(by=['clean_case', 'Chromosome', 'Start']).reset_index(drop=True)

#convert both X and Y to csv for viewing
df_X_final.to_csv('/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/filtered_lusc_X.csv', index=False)
df_exp_final.to_csv('/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/Y/filtered_lusc_Y.csv', index=False)






In [4]:
#Now that we have our filtered matrices copy number (X) and gene expression (Y) that match the TCGA IDs
#It was observed from the that there are multiple copy number variables we need to consider
#we have two options based on the estimators used
#For categorical representation of copy number (e.g 0=Homozygous Deletion, 3+= Gain/Amplification) this is best for Discrete Mutual Information/Likelihood Ratio Classifiers
#For continuous scalar representation of copy number (e.eg absolute copy number estimate or segment Log R Ratio), this is best suited for KDE, GMM, k-NN (kpN)
#Downstream we will utilize the data differently based on the total copy number or Copy Number column when implementing the estimators

#Lets first align the coordinates of the copy number data to a specific gene for each patient to connect the data or bridge the data together.
#segment level copy number data alone does not provide gene names or expression until we use the genomic coordinates to map them together.

#lets open the df_exp_final df and make the genes into a list
gene_list = df_exp_final['sample']

gene_list.to_csv('/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/Y/gene_list.csv', index=False)

#now that we have the list we went on e!Ensemble Biomart https://www.ensembl.org/biomart/martview to obtain the genes start and end (bp) and ensembl IDs
#lets load the exported tsv 

path_e_mart_export = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/mart_export.txt'

e_mart_export = pd.read_csv(path_e_mart_export, sep='\t')

#print(e_mart_export.head())

e_mart_export.to_csv('/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/mart_export.csv')



In [33]:
#Now that we have the gene start and end coordinates from the Y gene expression matrix, this section of code will work on organizing 
#the X CNV matrix 
#we will utilize a dictionary look up type of script instead of a row by row basis to determine the genes corresponding to the start end bp for the copy number variants
#we are using the chromosomes given by the e_mart gene data frame 
#we are also calculating the mean copy number per gene
    #this allows us to have a standard dimensionality for features X
    #also allows us to have biological alignment with gene expression Y (mapping copy number variants to corresponding gene locus allows us to analyze dosage-dependent effects 
    #like how the copy state of SOX4 drives transcription
    #we could later instead do a overlap-length weighted average or exon-evel/centermost exon granularity

import pandas as pd
import numpy as np

#lets name chromosome columns in both DataFrames
df_X_final = df_X_final.rename(columns={'Chromosome': 'chr_cnv'})
e_mart_export = e_mart_export.rename(columns={'Chromosome/scaffold name': 'chr_emart'})

#then define valid canonical chromosomes
valid_canonical_chroms = [str(i) for i in range(1, 23)] + ['X', 'Y', 'x', 'y']

#we will have a strict filter on BioMart data: data must be EXACTLY one of the valid canonical chromosomes (no patches/scaffolds)
e_mart_clean = e_mart_export[
    e_mart_export['chr_emart'].astype(str).str.strip().isin(valid_canonical_chroms)
].copy()
e_mart_clean['chr_emart_clean'] = e_mart_clean['chr_emart'].astype(str).str.strip()

#then clean CNV data: strip 'chr' prefix cleanly
df_X_final['chr_cnv_clean'] = df_X_final['chr_cnv'].astype(str).str.replace('chr', '', regex=False).str.strip()

#we will use a helper function for safe sorting of mixed chromosome names (1-22, X, Y)
def chrom_sort_key(x):
    return (0, int(x)) if str(x).isdigit() else (1, str(x))

print("BioMart clean chromosomes:", sorted(e_mart_clean['chr_emart_clean'].unique(), key=chrom_sort_key))
print("CNV clean chromosomes:", sorted(df_X_final['chr_cnv_clean'].unique(), key=chrom_sort_key))

#overlap mapping function of genes based on start end coordinates of df_X_final and e_mart_clean dfs
def map_segments_to_genes(df_segments, df_genes):
    mapped_rows = []

    for chrom, gene_group in df_genes.groupby('chr_emart_clean'):
        seg_group = df_segments[df_segments['chr_cnv_clean'] == chrom]
        if seg_group.empty:
            continue
            
        for _, gene in gene_group.iterrows():
            gene_name = gene['Gene name']
            g_start = gene['Gene start (bp)']
            g_end = gene['Gene end (bp)']
            
            #find overlapping segments using the coordinate space
            overlapping_segs = seg_group[
                (seg_group['Start'] <= g_end) & (seg_group['End'] >= g_start)
            ]
            
            if not overlapping_segs.empty:
                #then calculate mean copy number per patient for this gene
                patient_means = overlapping_segs.groupby('clean_case')['Copy_Number'].mean()
                
                #now create dictionary row containing metadata and patient values
                row_data = {'Chromosome': str(chrom), 'Gene name': gene_name}
                for patient_id, cn_val in patient_means.items():
                    row_data[patient_id] = cn_val
                    
                mapped_rows.append(row_data)
                
    if not mapped_rows:
        raise ValueError("No overlapping segments found! Check the chromosome formats.")
        
    #convert outside the loop so it aggregates all 24 chromosomes
    df_X_matrix = pd.DataFrame(mapped_rows)
    return df_X_matrix

#run the mapping function across the entire genome
df_X_gene_matrix = map_segments_to_genes(df_X_final, e_mart_clean)

#verify all chromosomes are present
print("Final Matrix chromosomes present:", sorted(df_X_gene_matrix['Chromosome'].unique(), key=chrom_sort_key))
print("Total genes mapped genome-wide:", len(df_X_gene_matrix))

#export df to CSV
output_path = '/rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/df_X_gene_matrix.csv'
df_X_gene_matrix.to_csv(output_path, index=False)
print(f"Successfully saved matrix to: {output_path}")



BioMart clean chromosomes: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']
CNV clean chromosomes: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']
Final Matrix chromosomes present: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']
Total genes mapped genome-wide: 17327
Successfully saved matrix to: /rds/projects/a/arjmandh-cin-it-gc-ha/Information Transmission from Copy Number Alterations to Gene Expression in Cancer/Data/X/df_X_gene_matrix.csv


In [35]:
#now that we have our two data frames almost ready for our estimator models. lets run a few checks. 
#we will compare all genes for both data sets as well as the IDs to see if they all match

import pandas as pd

#inspect X matrix
#extract features and IDs from X matrix
x_genes = set(df_X_gene_matrix['Gene name'])
#patient IDs are all columns except metadata columns
x_patients = set(df_X_gene_matrix.columns) - {'Chromosome', 'Gene name'}

print(f"--- X Matrix (Copy Number) ---")
print(f"Total genes: {len(x_genes):,}")
print(f"Total patients: {len(x_patients):,}\n")

#inspect Y matrix
y_genes = set(df_exp_final['sample'])



--- X Matrix (Copy Number) ---
Total genes: 17,140
Total patients: 486

